# iVAE on dSprites Dataset
This notebook is perfectly self-contained so you can run it on Google Colab with GPU. It contains dataset prep, model architectures, the training loop, and reconstruction visualization.

In [ ]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch import optim
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.optimize import linear_sum_assignment
from scipy.stats import spearmanr

# Check for GPU
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print('Using device:', device)

In [ ]:
# Download dSprites if not present
if not os.path.exists('dsprites.npz'):
    !wget https://github.com/deepmind/dsprites-dataset/raw/master/dsprites_ndarray_co1sh3sc6or40x32y32_64x64.npz -O dsprites.npz

class DSpritesDataset(Dataset):
    def __init__(self, root_dir='dsprites.npz', aux_factor='shape'):
        super().__init__()
        print(f"Loading dsprites from {root_dir}...")
        # Use mmap_mode='r' to memory-map the file, avoiding loading the entire 'imgs' array into RAM
        dataset_zip = np.load(root_dir, encoding='latin1', allow_pickle=True, mmap_mode='r')

        # Store the memmap object for images and load smaller latent variables into memory
        self._imgs_mmap = dataset_zip['imgs']
        self.latents_values = dataset_zip['latents_values']
        self.latents_classes = dataset_zip['latents_classes']
        self.num_samples = self._imgs_mmap.shape[0]

        self.s = torch.from_numpy(self.latents_values).float()

        factor_map = {'shape': 1, 'scale': 2, 'orientation': 3, 'posX': 4, 'posY': 5}
        idx = factor_map.get(aux_factor, 1)

        u_classes = self.latents_classes[:, idx]
        num_classes = len(np.unique(u_classes))

        u_onehot = np.zeros((self.num_samples, num_classes))
        u_onehot[np.arange(self.num_samples), u_classes] = 1

        self.u = torch.from_numpy(u_onehot).float()
        self.data_dim = 4096
        self.latent_dim = 6
        self.aux_dim = num_classes
        print(f"Dataset loaded. U: {self.u.shape}")

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        # Load image data on demand using the memory-mapped object
        img = self._imgs_mmap[idx]
        x_tensor = torch.from_numpy(img.reshape(self.data_dim)).float()
        return x_tensor, self.u[idx], self.s[idx]

dataset = DSpritesDataset(aux_factor='shape')
data_loader = DataLoader(dataset, batch_size=64, shuffle=True, drop_last=True)

In [ ]:
def weights_init(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight.data)

class MLP(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim, n_layers, activation='lrelu', slope=.1):
        super().__init__()
        self.n_layers = n_layers
        self.hidden_dim = [hidden_dim] * (self.n_layers - 1)
        
        self._act_f = []
        for _ in range(self.n_layers - 1):
            if activation == 'lrelu':
                self._act_f.append(lambda x: F.leaky_relu(x, negative_slope=slope))
        
        _fc_list = [nn.Linear(input_dim, self.hidden_dim[0])]
        for i in range(1, self.n_layers - 1):
            _fc_list.append(nn.Linear(self.hidden_dim[i - 1], self.hidden_dim[i]))
        _fc_list.append(nn.Linear(self.hidden_dim[-1], output_dim))
        self.fc = nn.ModuleList(_fc_list)

    def forward(self, x):
        h = x
        for c in range(self.n_layers):
            if c == self.n_layers - 1:
                h = self.fc[c](h)
            else:
                h = self._act_f[c](self.fc[c](h))
        return h

class DiscreteIVAE(nn.Module):
    def __init__(self, latent_dim, data_dim, aux_dim, n_layers=2, hidden_dim=20, activation='lrelu', slope=.1):
        super().__init__()
        self.prior_mean = torch.zeros(1).to(device)
        self.logl = MLP(aux_dim, latent_dim, hidden_dim, n_layers, activation=activation, slope=slope)
        self.f = MLP(latent_dim, data_dim, hidden_dim, n_layers, activation=activation, slope=slope)
        self.g = MLP(data_dim + aux_dim, latent_dim, hidden_dim, n_layers, activation=activation, slope=slope)
        self.logv = MLP(data_dim + aux_dim, latent_dim, hidden_dim, n_layers, activation=activation, slope=slope)
        self.apply(weights_init)

    def forward(self, x, u):
        # Encoder
        xu = torch.cat((x, u), 1)
        g = self.g(xu)
        logv = self.logv(xu)
        
        # Reparameterize
        std = torch.exp(0.5 * logv)
        eps = torch.randn_like(g)
        z = g + eps * std
        
        # Decoder
        f = torch.sigmoid(self.f(z))
        
        # Prior
        logl = self.logl(u)
        
        return f, (g, logv), z, (self.prior_mean, logl)

model = DiscreteIVAE(latent_dim=10, data_dim=4096, aux_dim=dataset.aux_dim, 
                     hidden_dim=1200, n_layers=4, activation='lrelu').to(device)
optimizer = optim.Adam(model.parameters(), lr=5e-4)

def calculate_mcc(z, s):
    """Calculates Mean Correlation Coefficient (MCC)
    z: predicted latents (N, D)
    s: true latents (N, D)
    """
    d = z.shape[1]
    # compute absolute spearman correlation matrix
    cc = np.zeros((d, d))
    for i in range(d):
        for j in range(d):
            cc[i, j] = np.abs(spearmanr(z[:, i], s[:, j])[0])
    
    # match sources using Hungarian algorithm
    row_ind, col_ind = linear_sum_assignment(-cc)
    mcc = cc[row_ind, col_ind].mean()
    return mcc


In [ ]:
epochs = 20
global_step = 0
latest_ckpt = 'ivae_dsprites_latest.pth'

start_epoch = 1
if os.path.exists(latest_ckpt):
    model.load_state_dict(torch.load(latest_ckpt, map_location=device))
    # Note: we don't store the exact epoch number in the file name anymore,
    # but we can try to infer it from logs or just start from 1 (weights are preserved)
    print(f"Resuming from {latest_ckpt}")
else:
    print("Starting training from scratch...")

for epoch in range(start_epoch, epochs + 1):
    model.train()
    train_loss, train_recon, train_kl = 0, 0, 0
    t0 = time.time()
    
    for i, (x, u, s_true) in enumerate(data_loader):
        x, u = x.to(device), u.to(device)
        optimizer.zero_grad()
        
        f, (g, logv), z, (h, logl) = model(x, u)
        
        recon_loss = F.binary_cross_entropy(f, x, reduction='sum')
        l = logl.exp()
        v = logv.exp()
        kl_loss = 0.5 * torch.sum(logl - logv - 1 + (g - h).pow(2) / l + v / l)
        
        loss = recon_loss + kl_loss  # Standard iVAE ELBO loss without artificial bottlenecks
        
        loss = loss / x.size(0)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        train_recon += (recon_loss.item() / x.size(0))
    train_kl /= len(data_loader)
    
    # Collect sample latents for MCC calculation (using 5000 random samples)
    model.eval()
    z_samples, s_samples = [], []
    count = 0
    with torch.no_grad():
        for x_m, u_m, s_m in data_loader:
            x_m, u_m = x_m.to(device), u_m.to(device)
            _, _, z_m, _ = model(x_m, u_m)
            z_samples.append(z_m.cpu().numpy())
            s_samples.append(s_m.cpu().numpy())
            count += x_m.size(0)
            if count >= 5000: break
    
    z_samples = np.concatenate(z_samples, axis=0)
    s_samples = np.concatenate(s_samples, axis=0)
    mcc_score = calculate_mcc(z_samples, s_samples)

    print(f"Epoch {epoch}/{epochs} | Time: {time.time()-t0:.1f}s | "
          f"Loss: {train_loss/len(data_loader):.4f} | Recon: {train_recon/len(data_loader):.4f} | "
          f"KL: {train_kl/len(data_loader):.4f} | MCC: {mcc_score:.4f}")
    
    # Save latest model checkpoint
    torch.save(model.state_dict(), latest_ckpt)


In [ ]:
# Visualize Reconstruction vs Original
model.eval()
with torch.no_grad():
    x_sample, u_sample, _ = next(iter(data_loader))
    x_sample, u_sample = x_sample.to(device), u_sample.to(device)
    
    f, _, _, _ = model(x_sample, u_sample)
    
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    for i in range(5):
        axes[0, i].imshow(x_sample[i].cpu().numpy().reshape(64, 64), cmap='gray')
        axes[0, i].set_title("Original")
        axes[0, i].axis('off')
        
        axes[1, i].imshow(f[i].cpu().numpy().reshape(64, 64), cmap='gray')
        axes[1, i].set_title("Reconstructed")
        axes[1, i].axis('off')
        
    plt.tight_layout()
    plt.show()